# Y-Shaped Tree Branch Prediction (UNET)

This notebook runs inference with the pre-trained UNET checkpoint from the project repository. It does **not** train the model.

## What you need
- A matching **RGB image** and **depth image** of the same scene.
- The repository's `unet/` model code.
- An internet connection so Colab can download the checkpoint.

The notebook resizes both inputs to **256 × 256**, applies RGB/depth preprocessing, predicts a branch probability map, then saves a binary mask and a red overlay.

**Input note:** the depth image is read as a single-channel grayscale image. Use the same kind of depth images that were used with the trained model.

## 1. Set up the environment

Run this cell first. It clones the public repository and installs the required packages.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/rkay0001-Monash/MMA3001-UNET-Model-for-Detecting-Y-Shaped-Tree-Branches.git"
REPO_DIR = Path("/content/MMA3001-UNET-Model-for-Detecting-Y-Shaped-Tree-Branches")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

%cd /content/MMA3001-UNET-Model-for-Detecting-Y-Shaped-Tree-Branches
%pip -q install numpy opencv-python-headless matplotlib

## 2. Download the pre-trained weights

The checkpoint is hosted as a GitHub Release asset, so it does not need to be committed to the repository.

In [ ]:
import urllib.request

WEIGHTS_URL = (
    "https://github.com/rkay0001-Monash/"
    "MMA3001-UNET-Model-for-Detecting-Y-Shaped-Tree-Branches/"
    "releases/download/v1.0.0/trained_weights.pth"
)
WEIGHTS_DIR = Path("/content/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_PATH = WEIGHTS_DIR / "trained_weights.pth"

if not WEIGHTS_PATH.exists() or WEIGHTS_PATH.stat().st_size == 0:
    print("Downloading checkpoint (about 355 MB). This may take a few minutes...")
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)

print(f"Checkpoint saved to: {WEIGHTS_PATH}")
print(f"Checkpoint size: {WEIGHTS_PATH.stat().st_size / (1024**2):.1f} MB")

## 3. Load the UNET model

The model definition and checkpoint format must match the training code.

In [ ]:
import torch
import sys
sys.path.insert(0, str(REPO_DIR))

from unet import UNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = UNet(n_channels=3, n_classes=1, bilinear=False)

try:
    checkpoint = torch.load(WEIGHTS_PATH, map_location=device, weights_only=True)
except TypeError:
    checkpoint = torch.load(WEIGHTS_PATH, map_location=device)

state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

print("Model loaded successfully.")

## 4. Upload the paired images

Upload **one RGB image** and **one matching depth image**. They should show the same scene and ideally have the same original dimensions. If you have several images, run this cell separately for each pair.

In [ ]:
from google.colab import files

print("Select the RGB image first.")
rgb_upload = files.upload()
if len(rgb_upload) != 1:
    raise ValueError("Please upload exactly one RGB image in this step.")
RGB_PATH = Path(next(iter(rgb_upload.keys())))

print("\nNow select the matching depth image.")
depth_upload = files.upload()
if len(depth_upload) != 1:
    raise ValueError("Please upload exactly one depth image in this step.")
DEPTH_PATH = Path(next(iter(depth_upload.keys())))

print("RGB image:", RGB_PATH)
print("Depth image:", DEPTH_PATH)

## 5. Preprocess the images

This follows the inference preprocessing: RGB is read as RGB, depth is read as grayscale, RGB is resized with area interpolation, depth with nearest-neighbour interpolation, depth is normalised by its maximum, and RGB is multiplied by the normalised depth image.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

FINAL_SIZE = 256
THRESHOLD = 0.5

rgb_bgr = cv2.imread(str(RGB_PATH), cv2.IMREAD_COLOR)
depth_raw = cv2.imread(str(DEPTH_PATH), cv2.IMREAD_GRAYSCALE)

if rgb_bgr is None:
    raise ValueError(f"Could not read RGB image: {RGB_PATH}")
if depth_raw is None:
    raise ValueError(f"Could not read depth image: {DEPTH_PATH}")
if rgb_bgr.shape[:2] != depth_raw.shape[:2]:
    raise ValueError(
        "RGB and depth image dimensions do not match: "
        f"RGB={rgb_bgr.shape[:2]}, depth={depth_raw.shape[:2]}. "
        "Use matching image pairs."
    )
if not np.any(depth_raw):
    raise ValueError("The depth image contains only zero values.")

rgb = cv2.cvtColor(rgb_bgr, cv2.COLOR_BGR2RGB)
rgb_resized = cv2.resize(rgb, (FINAL_SIZE, FINAL_SIZE), interpolation=cv2.INTER_AREA)
depth_resized = cv2.resize(depth_raw, (FINAL_SIZE, FINAL_SIZE), interpolation=cv2.INTER_NEAREST)

depth_norm = depth_resized.astype(np.float32)
depth_norm /= float(depth_norm.max())

# Soft depth masking, matching the project's inference workflow.
rgb_masked = np.clip(
    rgb_resized.astype(np.float32) * depth_norm[..., None], 0, 255
).astype(np.uint8)

input_array = rgb_masked.astype(np.float32) / 255.0
input_tensor = torch.from_numpy(input_array.transpose(2, 0, 1)).unsqueeze(0).to(device)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(rgb_resized)
axes[0].set_title("RGB (resized)")
axes[1].imshow(depth_norm, cmap="gray")
axes[1].set_title("Depth (normalised)")
axes[2].imshow(rgb_masked)
axes[2].set_title("RGB × depth")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Predict the branch mask

The model output is treated as logits, converted to probabilities with sigmoid, and thresholded at 0.5 to create a binary mask.

In [ ]:
with torch.no_grad():
    logits = model(input_tensor)
    probabilities = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()

binary_mask = (probabilities >= THRESHOLD).astype(np.uint8) * 255

overlay = rgb_resized.copy()
mask_bool = binary_mask > 0
red = np.zeros_like(overlay)
red[..., 0] = 255
alpha = 0.45
overlay[mask_bool] = (
    (1 - alpha) * overlay[mask_bool].astype(np.float32)
    + alpha * red[mask_bool].astype(np.float32)
).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(probabilities, cmap="viridis", vmin=0, vmax=1)
axes[0].set_title("Predicted probability")
axes[1].imshow(binary_mask, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Binary mask (threshold={THRESHOLD})")
axes[2].imshow(overlay)
axes[2].set_title("Mask overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Save and download the outputs

Three PNG files are saved: the probability map, binary mask, and red overlay. They are bundled into one ZIP for convenience.

In [ ]:
import zipfile
from google.colab import files

OUTPUT_DIR = Path("/content/tree_branch_predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

stem = RGB_PATH.stem
probability_path = OUTPUT_DIR / f"{stem}_probability.png"
mask_path = OUTPUT_DIR / f"{stem}_mask.png"
overlay_path = OUTPUT_DIR / f"{stem}_overlay.png"
zip_path = Path("/content/tree_branch_predictions.zip")

probability_img = np.clip(probabilities * 255.0, 0, 255).astype(np.uint8)
cv2.imwrite(str(probability_path), probability_img)
cv2.imwrite(str(mask_path), binary_mask)
cv2.imwrite(str(overlay_path), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (probability_path, mask_path, overlay_path):
        archive.write(path, arcname=path.name)

print("Saved files:")
print(probability_path)
print(mask_path)
print(overlay_path)
print(zip_path)
files.download(str(zip_path))

## Notes and troubleshooting

- **CUDA out of memory:** In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU), then rerun the notebook from the beginning. CPU inference should also work but may be slower.
- **Checkpoint loading error:** Confirm that the repository's `unet/` files correspond to the architecture used to train this checkpoint.
- **Poor predictions:** Use a matching RGB/depth pair and the same depth-image format as the training pipeline. Depth masking and 256 × 256 resizing affect predictions.
- **Output size:** Predictions are 256 × 256 because the model input is resized to that size; this notebook does not scale the mask back to the original image dimensions.
- This notebook performs inference only; it does not retrain or fine-tune the model.